In [5]:
import sys
import os
import numpy as np

# Them duong dan tro ve python-rag-service de co the import package 'app'
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, "python-rag-service"))
if not os.path.isdir(PROJECT_ROOT):
    PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, "..", "python-rag-service"))
if not os.path.isdir(PROJECT_ROOT):
    raise FileNotFoundError("Khong tim thay thu muc python-rag-service. Hay mo notebook tu thu muc Chatbot-medical.")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from langchain_huggingface import HuggingFaceEmbeddings
from app.settings import settings
from app.utils.reranker import Reranker

def calculate_chunk_scores(query: str, chunk_text: str):
    print("=" * 60)
    print(f"QUERY: {query}")
    print(f"CHUNK: {chunk_text}")
    print("=" * 60)
    
    # -------------------------------------------------------------
    # 1. DENSE SCORE (Cosine Similarity bang Vector Embeddings)
    # -------------------------------------------------------------
    try:
        print("\n[1] Dang tinh Dense Score...")
        embeddings = HuggingFaceEmbeddings(
            model_name=settings.embedding_model,
            model_kwargs={'trust_remote_code': True}
        )
        query_vector = embeddings.embed_query(query)
        chunk_vector = embeddings.embed_query(chunk_text)
        
        dot_product = np.dot(query_vector, chunk_vector)
        norm_q = np.linalg.norm(query_vector)
        norm_c = np.linalg.norm(chunk_vector)
        
        dense_score = (dot_product / (norm_q * norm_c)) if (norm_q * norm_c) != 0 else 0.0
        print(f"--> [Dense Score]: {dense_score:.4f}")
    except Exception as e:
        print(f"--> [Dense Score] Loi: {e}")

    # -------------------------------------------------------------
    # 2. RERANKER SCORE (Cross-Encoder / BGE-Reranker-M3)
    # -------------------------------------------------------------
    try:
        print("\n[2] Dang tinh Reranker Score...")
        reranker = Reranker()
        candidates = [{"chunk_text": chunk_text}]
        
        results = reranker.rerank_candidates(query, candidates, top_n=1)
        reranker_score = results[0]["score_cross_encoder"] if results else 0.0
        print(f"--> [Reranker Score]: {reranker_score:.4f}")
    except Exception as e:
        print(f"--> [Reranker Score] Loi: {e}")
        
    print("\n" + "=" * 60)

if __name__ == "__main__":
    # Dien gia tri query va chunk_text test vao day
    test_query = "Triệu chứng lâm sàng của bệnh ung thư bạch cầu cấp lympho là gì? "
    test_chunk = """LÂM SÀNG
Biểu hiện không đặc hiệu, khởi phát bệnh một vài tuần đến một vài tháng.
Mệt mỏi, chán ăn, sốt kéo dài, ra nhiều mồ hôi ban đêm, nhiễm trùng khó điều trị, thiếu
máu, xuất huyết dưới da hoặc niêm mạc, gan, lách, hạch to, đau xương hoặc khớp. Biểu
hiện hiếm gặp hơn: tăng áp lực nội sọ, liệt dây thần kinh sọ, khó thở do u trung thất, tinh
hoàn to."""
    
    calculate_chunk_scores(query=test_query, chunk_text=test_chunk)


QUERY: Triệu chứng lâm sàng của bệnh ung thư bạch cầu cấp lympho là gì? 
CHUNK: LÂM SÀNG
Biểu hiện không đặc hiệu, khởi phát bệnh một vài tuần đến một vài tháng.
Mệt mỏi, chán ăn, sốt kéo dài, ra nhiều mồ hôi ban đêm, nhiễm trùng khó điều trị, thiếu
máu, xuất huyết dưới da hoặc niêm mạc, gan, lách, hạch to, đau xương hoặc khớp. Biểu
hiện hiếm gặp hơn: tăng áp lực nội sọ, liệt dây thần kinh sọ, khó thở do u trung thất, tinh
hoàn to.

[1] Dang tinh Dense Score...
--> [Dense Score]: 0.5449

[2] Dang tinh Reranker Score...
--> [Reranker Score]: 0.5761

